# 🧪 W5-D3 概念实验：DeepSeek-R1 式推理模型原理

> 配套阅读：`第5周-Day3-DeepSeek-R1推理模型原理.md`（R1-Zero/R1、MLA、Reasoning Token 在那边）
>
> 本 notebook 不调用任何模型，用可运行的模拟回答四个问题：
> 1. **GRPO 为什么不需要额外价值模型（critic）？** —— 组内相对优势的可执行实现
> 2. **RL 训练中"推理行为"如何涌现？** —— R1-Zero 式训练模拟（含 aha moment）
> 3. **Reasoning Token 长什么样、成本怎么算？** —— `<think>` 段解析与统计
> 4. **R1 什么时候更强、什么时候反而吃亏？**

环境：仅 numpy / 标准库 / matplotlib，本地模拟，无网络。

## 实验 1：GRPO 组相对优势（纯 numpy）

GRPO 的核心：对同一道题采样 G 个回答，用**组内均值**当 baseline、**组内标准差**归一化，
得到每个回答的优势 A_i —— 好于组内平均的为正、差的为负。
不需要像 PPO 那样再训练一个 critic 网络去估 V(s)，省一半显存与一套训练流水线。

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

def grpo_advantage(rewards):
    """一组 G 个回答的 reward → 组内标准化优势（GRPO 式，无 critic）。"""
    rewards = np.asarray(rewards, dtype=float)
    baseline = rewards.mean()                  # 组内均值代替 critic 的 V(s)
    std = rewards.std() + 1e-8
    return (rewards - baseline) / std          # 正=好于组内平均，负=差

# 一个 batch：2 道题，每题采样 G=8 个回答
# reward = 正确性(1/0) + 格式奖励(0.1) - 长度惩罚(每千 token 0.05)
problems = {
    "P1（较易）": dict(p_correct=[0.75, 0.65, 0.85, 0.55, 0.70, 0.80, 0.60, 0.72],
                       lengths   =[420, 380, 900, 300, 500, 650, 350, 480]),
    "P2（较难）": dict(p_correct=[0.25, 0.15, 0.35, 0.10, 0.20, 0.40, 0.05, 0.30],
                       lengths   =[1500, 900, 2200, 700, 1300, 2600, 600, 1800]),
}
for name, info in problems.items():
    correct = (rng.random(8) < np.array(info["p_correct"])).astype(float)
    fmt = (rng.random(8) < 0.9) * 0.1
    length_pen = -np.array(info["lengths"]) / 1000 * 0.05
    rewards = correct + fmt + length_pen
    adv = grpo_advantage(rewards)
    print(f"== {name} ==")
    for i in range(8):
        tag = "✓对" if correct[i] else "✗错"
        print(f"  回答{i+1}: {tag} len={info['lengths'][i]:>4}  reward={rewards[i]:+.3f}  adv={adv[i]:+.2f}")
    print(f"  → 组均值={rewards.mean():.3f}（作为 baseline，替代 PPO 的 critic）")
print()
print("解读：即使全组都对/全错，标准化后优势≈0，这组样本不产生梯度——")
print("GRPO 只关心'组内谁比谁好'，天然适合'结果可判定对错'的数学/代码任务。")

## 实验 2：R1-Zero 式训练模拟 —— 推理行为的涌现

用一个 4 动作策略（直接答 / 试错 / 回头反思 / 验证）模拟纯 RL 训练：
- reward = 答对(1) + 格式(0.1) − 长度惩罚，**不监督过程**，只看结果
- 每步：采样一组回答 → GRPO 优势 → 优势加权更新策略概率

观察：答对率上升的同时，**"反思/验证"行为自发增多**（aha moment 的影子），
平均长度先增后稳——因为没有过程监督，这些行为是被结果奖励"养"出来的。

In [ ]:
ACTIONS = ["直接答", "试错推进", "回头反思", "验证检查"]

def run_rl(steps=400, G=16, seed=3):
    rng = np.random.default_rng(seed)
    policy = np.array([0.55, 0.25, 0.10, 0.10])   # 初始策略：偏直接答
    hist = []
    for t in range(steps):
        q_diff = rng.uniform(0.1, 0.85)           # 每批题目难度不同：难题养反思，易题养直接答
        acts = rng.choice(4, size=G, p=policy)
        # 每个回答的'解题力'：直接答弱、试错中、反思/验证强（叠加随机扰动）
        power = np.array([0.35, 0.6, 0.85, 0.9])[acts] * (0.8 + 0.4 * rng.random(G))
        correct = rng.random(G) < (power * (1 - q_diff) + power**2 * q_diff)
        lens = 200 + 260 * acts + 180 * rng.random(G)          # 反思/验证 → 更长
        rewards = correct + 0.1 - lens / 1000 * 0.05
        adv = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
        # 策略更新：优势大的动作概率上升（指数加权 + 平滑，避免过早坍缩）
        score = np.array([(adv[acts == a]).sum() for a in range(4)])
        new_p = policy * np.exp(0.05 * score)
        new_p /= new_p.sum()
        policy = 0.88 * policy + 0.12 * new_p
        policy /= policy.sum()
        hist.append((t, correct.mean(), lens.mean(), policy.copy()))
    return policy, hist

final_policy, hist = run_rl()
h = np.array([(t, acc, ln) for t, acc, ln, _ in hist])
refl = np.array([p[2] + p[3] for *_, p in hist])   # 反思+验证 概率

print(f"初始策略: 直接答55% | 试错25% | 反思10% | 验证10%")
print(f"训练{len(hist)}步后: " + " | ".join(f"{ACTIONS[i]}{final_policy[i]:.0%}" for i in range(4)))
print(f"组内答对率: {h[0,1]:.0%} → {h[-1,1]:.0%}   平均长度: {h[0,2]:.0f} → {h[-1,2]:.0f} tokens")
first = int(np.argmax(refl > 0.5))
print(f"反思+验证概率超过 50% 的训练步: ≈第 {first} 步（行为'涌现'的时机）")

## 实验 3：Reasoning Token —— `<think>` 段解析与成本账

R1 把思考过程显式放进 `<think>...</think>`。这里写一个真实的解析器 + 粗粒度分词器
（中文≈1.5 字/token 的经验比例），统计：简单题 vs 复杂题的思考/回答 token 分布，
以及"思考成本占比"——这就是 API 计费里 reasoning token 的账。

In [ ]:
import re

def tokenize_len(text):
    """粗粒度 token 估计：ASCII 词算 1，中文按 1.5 字/token。"""
    ascii_words = len(re.findall(r"[A-Za-z0-9_]+", text))
    cjk_chars = len(re.findall(r"[\u4e00-\u9fff]", text))
    return int(ascii_words + cjk_chars / 1.5)

def parse_response(text):
    m = re.search(r"<think>(.*?)</think>\s*(.*)", text, re.S)
    think, answer = (m.group(1), m.group(2)) if m else ("", text)
    return tokenize_len(think), tokenize_len(answer)

samples = {
 "简单题": ["<think>用户问省会，直接答。</think>广东省的省会是广州。",
            "<think>常识题，无需展开。</think>水的沸点是100℃（标准大气压）。"],
 "复杂题": ["<think>设原价 x。打8折后是0.8x，再减50元券：0.8x-50=70 ⇒ x=150。验证：150*0.8=120，120-50=70 ✓</think>衬衫原价150元。",
            "<think>先分解子问题：1) 求库存周转率=销量/平均库存；2) 求两个月的平均库存；3) 代入验算。销量 240+300=540；平均库存 (80+70)/2=75；周转率=540/75=7.2。再验算一遍量纲 ✓</think>库存周转率为 7.2 次/月。"],
}
for kind, texts in samples.items():
    rows = [parse_response(t) for t in texts]
    for t, (think_n, ans_n) in zip(texts, rows):
        pass
    think_avg = np.mean([r[0] for r in rows]); ans_avg = np.mean([r[1] for r in rows])
    share = think_avg / (think_avg + ans_avg)
    print(f"{kind}: 思考≈{think_avg:.0f} tok | 答案≈{ans_avg:.0f} tok | 思考成本占比 {share:.0%}")
print("\n解读：推理模型的'思考'是计费且计时的——复杂题占比可到 70%+，")
print("简单题也会'想一下'。=> 混合策略：路由器把简单题发给快模型（见实验4）。")

## 实验 4：可视化 —— 训练涌现曲线 + R1 vs 普通模型选型

In [ ]:
# matplotlib 中文字体配置（NotoSansCJK，每次画图前先跑这段）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = fontManager_font = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

# 左图：R1-Zero 训练模拟曲线
axes[0].plot(h[:, 0], h[:, 1] * 100, color="#219ebc", label="组内答对率")
axes[0].plot(h[:, 0], h[:, 2] / 30, color="#fb8500", alpha=0.8, label="平均长度/30")
axes[0].plot(h[:, 0], refl * 100, color="#8e7dbe", label="反思+验证概率")
axes[0].set_xlabel("训练步"); axes[0].set_ylabel("%（长度已缩放）")
axes[0].set_title("R1-Zero 模拟：正确率↑ 长度↑ 反思行为涌现")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# 右图：R1 vs 普通模型（模拟评测）
tasks = ["事实问答", "翻译", "多步数学", "代码调试", "逻辑推理"]
normal = [88, 92, 52, 48, 55]
r1     = [89, 90, 84, 80, 86]
x = np.arange(len(tasks)); w = 0.38
axes[1].bar(x - w/2, normal, w, label="普通模型", color="#adb5bd")
axes[1].bar(x + w/2, r1, w, label="R1 式推理模型", color="#fb8500")
axes[1].set_xticks(x); axes[1].set_xticklabels(tasks, fontsize=9)
axes[1].set_ylabel("评测准确率 (%)"); axes[1].set_ylim(40, 100)
axes[1].set_title("R1 并非全场景更强（且思考 token 更贵更慢）")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout(); plt.show()
print("右图解读：推理模型在多步/逻辑任务大幅领先；在简单任务≈持平甚至略慢略贵。")
print("=> 工程上用'难度路由'混用两类模型（md 的混合模型策略）。")

## 小结

- **GRPO**：组内均值当 baseline、组内标准化当优势 → 省掉 critic
- **R1-Zero**：只给结果奖励，反思/验证等推理行为可以被"养"出来（涌现）
- **Reasoning Token**：看得见、可解析、要计费 —— 复杂题思考占比 70%+
- **选型**：R1 不是全能替身，按任务难度路由才是正解